# DBKala — Sales Analytics & EDA

This notebook explores sales performance using data extracted
from the DBKala PostgreSQL database.

## Objectives

- Understand the structure and quality of sales data
- Analyze revenue and profit
- Identify sales trends over time
- Analyze product and category performance
- Explore payment methods and order status
- Identify anomalies and data quality issues

In [2]:
import pandas as pd
import numpy as np
pd.set_option("display.max_columns", None)

from pathlib import Path
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine

In [3]:
load_dotenv("../.env")

True

In [4]:
DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")

### Create Connection Engine to Database

In [5]:
engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

In [6]:
with engine.connect() as connection:
    print("Database connection successful!")

Database connection successful!


## Extract Data

In [7]:
query_path = Path("../sql/sales_query.sql")

query = query_path.read_text()

sales_df = pd.read_sql(query, engine)

## Getting some Insight

In [8]:
sales_df.head()

,order_id,customer_id,order_date,order_status,order_priority,payment_method,quantity,return_status,final_price_at_order_time,cost_price_at_order_time,branch_id,discount,product_id,product_name,category_id,category_name
0,25263,1599,2026-02-01,Received,High,In-App Wallet,5,None,1330.92,78.83,15,0.25,358,NitroBlade Ultra,28,Laptops
1,25263,1599,2026-02-01,Received,High,In-App Wallet,5,None,1330.92,53.99,20,0.25,358,NitroBlade Ultra,28,Laptops
2,25263,1599,2026-02-01,Received,High,In-App Wallet,3,None,1330.92,61.28,13,0.25,358,NitroBlade Ultra,28,Laptops
3,25263,1599,2026-02-01,Received,High,In-App Wallet,4,None,1330.92,78.12,16,0.25,358,NitroBlade Ultra,28,Laptops
4,25263,1599,2026-02-01,Received,High,In-App Wallet,5,None,1330.92,75.55,11,0.25,358,NitroBlade Ultra,28,Laptops


In [17]:
print(f"Sales dataset shape = {sales_df.shape}", end='\n')

print("Sales dataset info :", end='\n')
print(sales_df.info())

Sales dataset shape = (149101, 16)
Sales dataset info :
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 149101 entries, 0 to 149100
Data columns (total 16 columns):
 #   Column                     Non-Null Count   Dtype  
---  ------                     --------------   -----  
 0   order_id                   149101 non-null  int64  
 1   customer_id                149101 non-null  int64  
 2   order_date                 149101 non-null  object 
 3   order_status               149101 non-null  object 
 4   order_priority             149101 non-null  object 
 5   payment_method             149101 non-null  object 
 6   quantity                   149101 non-null  int64  
 7   return_status              0 non-null       object 
 8   final_price_at_order_time  149101 non-null  float64
 9   cost_price_at_order_time   149101 non-null  float64
 10  branch_id                  149101 non-null  int64  
 11  discount                   149101 non-null  float64
 12  product_id                 149

In [19]:
print(f"Sum of null values = \n {sales_df.isna().sum()}")

Sum of null values = 
 order_id                          0
customer_id                       0
order_date                        0
order_status                      0
order_priority                    0
payment_method                    0
quantity                          0
return_status                149101
final_price_at_order_time         0
cost_price_at_order_time          0
branch_id                         0
discount                          0
product_id                        0
product_name                      0
category_id                       0
category_name                     0
dtype: int64


In [ ]:
print(f"Sum of duplicated values = {sales_df.duplicated().sum()}")

Sum of dduplicated values = 0


In [23]:
sales_df.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
order_id,149101.0,NaN,NaN,NaN,14792.708895,8285.512288,1.0,7555.0,15079.0,22633.0,26864.0
customer_id,149101.0,NaN,NaN,NaN,992.899223,568.729085,1.0,502.0,997.0,1478.0,1987.0
order_date,149101,2026,2026-02-20,842,NaN,NaN,NaN,NaN,NaN,NaN,NaN
order_status,149101,4,Received,144302,NaN,NaN,NaN,NaN,NaN,NaN,NaN
order_priority,149101,5,High,49566,NaN,NaN,NaN,NaN,NaN,NaN,NaN
payment_method,149101,5,In-App Wallet,55592,NaN,NaN,NaN,NaN,NaN,NaN,NaN
quantity,149101.0,NaN,NaN,NaN,5.254532,2.711831,1.0,3.0,5.0,8.0,10.0
return_status,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
final_price_at_order_time,149101.0,NaN,NaN,NaN,541.29158,538.038437,2.55,85.69,249.76,1052.01,2910.0
cost_price_at_order_time,149101.0,NaN,NaN,NaN,316.310389,394.615289,-426.39,63.22,118.68,452.4,2672.48


## Feature Engineering

At this part we craete some economic features.

- *revenue* :
- *cost* :
- *profit* :
- *profit_margin* :

In [9]:
sales_df["revenue"] = (
    sales_df["quantity"] * sales_df["final_price_at_order_time"]
)

sales_df["cost"] = (
    sales_df["quantity"] * sales_df["cost_price_at_order_time"]
)

sales_df["profit"] = (
    sales_df["revenue"] - sales_df["cost"]
)

sales_df["profit_margin"] = (
    sales_df["profit"] / sales_df["revenue"]
) * 100

In [10]:
sales_df[[
    "revenue",
    "cost",
    "profit",
    "profit_margin"
]].describe().T

,count,mean,std,min,25%,50%,75%,max
revenue,149101.0,2653.708248,3015.958007,2.550000,378.920000,1242.060000,4169.400000,28500.000000
cost,149101.0,1723.796406,2623.175062,-2984.730000,252.880000,548.440000,1885.280000,24099.300000
profit,149101.0,929.911842,2029.222988,-8737.100000,-37.200000,66.680000,775.710000,11288.100000
profit_margin,149101.0,17.335762,41.528041,-112.577065,-8.495638,12.165406,33.011026,225.512295


In [11]:
#convert type of order_date into datetime
sales_df["order_date"] = pd.to_datetime(sales_df["order_date"])

# split order_date to some time features
sales_df["year"] = sales_df["order_date"].dt.year 
sales_df["month"] = sales_df["order_date"].dt.month
sales_df["month_name"] = sales_df["order_date"].dt.month_name()

sales_df["year_month"] = (
    sales_df["order_date"]
    .dt.to_period("M")
    .astype(str)
)

## Data Quality Investigation

### Check unusual *dates*

In [12]:
sales_df[
    sales_df["order_date"].dt.year > 2026
][[
    "order_id",
    "order_date",
    "customer_id",
    "product_name",
    "quantity",
    "final_price_at_order_time",
    "cost_price_at_order_time"
]]

,order_id,order_date,customer_id,product_name,quantity,final_price_at_order_time,cost_price_at_order_time
28338,774,2099-10-22,344,FunPrint Jeans,5,22.33,31.95
28339,774,2099-10-22,344,WinterCozy Dress,10,94.48,92.91
89775,13232,2099-02-27,142,WindGuard Bottle,10,250.86,342.55


### Check the amount of *quantity* 

In [14]:
print(sales_df["quantity"].describe())

print("Zero quantity:",
      (sales_df["quantity"] == 0).sum())

print("Negative quantity:",
      (sales_df["quantity"] < 0).sum())

count    149101.000000
mean          5.254532
std           2.711831
min           1.000000
25%           3.000000
50%           5.000000
75%           8.000000
max          10.000000
Name: quantity, dtype: float64
Zero quantity: 0
Negative quantity: 0


### Check *prices*

In [15]:
sales_df[
    ["final_price_at_order_time", "cost_price_at_order_time"]
].describe()

,final_price_at_order_time,cost_price_at_order_time
count,149101.000000,149101.000000
mean,541.291580,316.310389
std,538.038437,394.615289
min,2.550000,-426.390000
25%,85.690000,63.220000
50%,249.760000,118.680000
75%,1052.010000,452.400000
max,2910.000000,2672.480000


In [16]:
print(
    "Negative selling prices:",
    (sales_df["final_price_at_order_time"] < 0).sum()
)

print(
    "Negative cost prices:",
    (sales_df["cost_price_at_order_time"] < 0).sum()
)

print(
    "Zero selling prices:",
    (sales_df["final_price_at_order_time"] == 0).sum()
)

print(
    "Zero cost prices:",
    (sales_df["cost_price_at_order_time"] == 0).sum()
)

Negative selling prices: 0
Negative cost prices: 2
Zero selling prices: 0
Zero cost prices: 0


In [27]:
sales_df[
    sales_df["cost_price_at_order_time"] < 0
][[
    "order_id",
    "order_date",
    "product_name",
    "quantity",
    "final_price_at_order_time",
    "cost_price_at_order_time",
    "discount",
    "branch_id"
]]

,order_id,order_date,product_name,quantity,final_price_at_order_time,cost_price_at_order_time,discount,branch_id
35880,2285,2020-07-05,GoPro Alpha,7,804.82,-426.39,0.24,15
118007,18980,2024-01-20,CozyNest Plant,10,185.44,-232.75,0.26,11


In [28]:
negative_profit = (sales_df["profit"] < 0).sum()
positive_profit = (sales_df["profit"] > 0).sum()
zero_profit = (sales_df["profit"] == 0).sum()

print("Negative profit:", negative_profit)
print("Positive profit:", positive_profit)
print("Zero profit:", zero_profit)

print(
    "Negative profit percentage:",
    round(
        (negative_profit / len(sales_df)) * 100,
        2
    ),
    "%"
)

Negative profit: 50393
Positive profit: 98688
Zero profit: 20
Negative profit percentage: 33.8 %


In [56]:
loss_df = sales_df[sales_df["profit"] < 0]

loss_df[[
    "product_name",
    "category_name",
    "quantity",
    "final_price_at_order_time",
    "cost_price_at_order_time",
    "revenue",
    "cost",
    "profit"
]].head(10)

,product_name,category_name,quantity,final_price_at_order_time,cost_price_at_order_time,revenue,cost,profit
24000,LittleStar Sweater,Kids,1,40.55,86.2,40.55,86.2,-45.65
24001,LittleStar Sweater,Kids,1,40.55,86.2,40.55,86.2,-45.65
24002,LittleStar Sweater,Kids,1,40.55,86.2,40.55,86.2,-45.65
24003,LittleStar Sweater,Kids,1,40.55,86.2,40.55,86.2,-45.65
24004,LittleStar Sweater,Kids,1,40.55,86.2,40.55,86.2,-45.65
24005,LittleStar Sweater,Kids,1,40.55,86.2,40.55,86.2,-45.65
24006,LittleStar Sweater,Kids,1,40.55,86.2,40.55,86.2,-45.65
24007,LittleStar Sweater,Kids,1,40.55,86.2,40.55,86.2,-45.65
24008,LittleStar Sweater,Kids,1,40.55,86.2,40.55,86.2,-45.65
24009,LittleStar Sweater,Kids,1,40.55,86.2,40.55,86.2,-45.65


In [66]:
little_star = sales_df[
    sales_df["product_name"] == "LittleStar Sweater"
].copy()

print(little_star.shape)
print(little_star["order_id"].nunique())
print(
    little_star[
    [
        "product_name",
        "quantity",
        "final_price_at_order_time",
        "cost_price_at_order_time",
        "profit"
    ]
].duplicated().sum()
)

(616, 24)
616
401


In [67]:
little_star.groupby(
    [
        "product_name",
        "quantity",
        "final_price_at_order_time",
        "cost_price_at_order_time"
    ]
).agg(
    occurrences=("order_id", "size"),
    unique_orders=("order_id", "nunique"),
    unique_customers=("customer_id", "nunique"),
    unique_branches=("branch_id", "nunique")
).sort_values(
    "occurrences",
    ascending=False
).head(10)

occurrences  \
product_name       quantity final_price_at_order_time cost_price_at_order_time                
LittleStar Sweater 1        40.55                     86.20                             402   
                            16.52                     14.89                               1   
                            11.13                     9.57                                1   
                            18.88                     15.41                               1   
                            22.24                     23.13                               1   
                            23.02                     20.22                               1   
                            18.48                     21.23                               1   
                            24.49                     23.09                               1   
                            25.74                     33.00                               1   
                            27.57                     42.57                               1   

                                                                                unique_orders  \
product_name       quantity final_price_at_order_time cost_price_at_order_time                  
LittleStar Sweater 1        40.55                     86.20                               402   
                            16.52                     14.89                                 1   
                            11.13                     9.57                                  1   
                            18.88                     15.41                                 1   
                            22.24                     23.13                                 1   
                            23.02                     20.22                                 1   
                            18.48                     21.23                                 1   
                            24.49                     23.09                                 1   
                            25.74                     33.00                                 1   
                            27.57                     42.57                                 1   

                                                                                unique_customers  \
product_name       quantity final_price_at_order_time cost_price_at_order_time                     
LittleStar Sweater 1        40.55                     86.20                                  402   
                            16.52                     14.89                                    1   
                            11.13                     9.57                                     1   
                            18.88                     15.41                                    1   
                            22.24                     23.13                                    1   
                            23.02                     20.22                                    1   
                            18.48                     21.23                                    1   
                            24.49                     23.09                                    1   
                            25.74                     33.00                                    1   
                            27.57                     42.57                                    1   

                                                                                unique_branches  
product_name       quantity final_price_at_order_time cost_price_at_order_time                   
LittleStar Sweater 1        40.55                     86.20                                   1  
                            16.52                     14.89                                   1  
                            11.13                     9.57                                    1  
                            18.88                     15.41                        

In [69]:
little_star_402 = sales_df[
    (sales_df["product_name"] == "LittleStar Sweater") &
    (sales_df["quantity"] == 1) &
    (sales_df["final_price_at_order_time"] == 40.55) &
    (sales_df["cost_price_at_order_time"] == 86.20)
].copy()

little_star_402[
    [
        "order_id",
        "customer_id",
        "order_date",
        "branch_id",
        "discount",
        "payment_method",
        "profit"
    ]
].sort_values("order_date").head(5)

,order_id,customer_id,order_date,branch_id,discount,payment_method,profit
24000,26463,486,2026-02-20,11,0.25,BNPL,-45.65
24273,26736,1425,2026-02-20,11,0.25,BNPL,-45.65
24272,26735,358,2026-02-20,11,0.25,BNPL,-45.65
24271,26734,664,2026-02-20,11,0.25,BNPL,-45.65
24270,26733,905,2026-02-20,11,0.25,BNPL,-45.65


In [70]:
print(little_star_402["order_date"].min(), little_star_402["order_date"].max())
print(little_star_402["branch_id"].value_counts())

2026-02-20 00:00:00 2026-02-20 00:00:00
branch_id
11    402
Name: count, dtype: int64


**Anomaly detected:**

402 transactions for *LittleStar Sweater* occurred on a single day at a selling price significantly below cost, generating approximately 18.35K in losses. This single event materially distorted the product's overall profitability.

In [42]:
product_analysis = (
    sales_df
    .groupby("product_name")
    .agg(
        total_revenue=("revenue", "sum"),
        total_cost=("cost", "sum"),
        total_profit=("profit", "sum"),
        units_sold=("quantity", "sum"),
        transactions=("order_id", "count"),
        loss_transactions=("profit", lambda x: (x < 0).sum())
    )
)

product_analysis["loss_rate"] = (
    product_analysis["loss_transactions"]
    / product_analysis["transactions"]
    * 100
)

product_analysis.sort_values(
    "total_profit"
).head(10)

,total_revenue,total_cost,total_profit,units_sold,transactions,loss_transactions,loss_rate
product_name,,,,,,,
LittleStar Sweater,51668.04,66310.10,-14642.06,1558,616,480,77.922078
Breeze Jacket,47194.44,47594.31,-399.87,982,170,77,45.294118
SportFlex Tee,43277.36,42695.73,581.63,904,163,78,47.852761
HappyBear Tee,38255.79,37383.36,872.43,1261,231,97,41.991342
ModernWear Shorts,48928.43,47942.43,986.00,993,169,81,47.928994
LittleStar Tee,32731.58,31607.76,1123.82,1062,198,80,40.404040
LeatherCraft Jeans,36382.51,35113.10,1269.41,809,154,72,46.753247
LittleStar Hoodie,37982.02,36655.16,1326.86,1216,218,99,45.412844
Alpine Pants,41660.21,40224.65,1435.56,873,156,61,39.102564


In [45]:
category_analysis = (
    sales_df
    .groupby("category_name")
    .agg(
        revenue=("revenue", "sum"),
        cost=("cost", "sum"),
        profit=("profit", "sum"),
        units_sold=("quantity", "sum"),
        transactions=("order_id", "count")
    )
    .sort_values("profit")
)

category_analysis

,revenue,cost,profit,units_sold,transactions
category_name,,,,,
Kids,1.820118e+06,1719385.42,1.007328e+05,56973,10676
Men,2.781839e+06,2601808.58,1.800307e+05,56959,10331
Women,3.526958e+06,3290518.32,2.364399e+05,58741,10654
Fitness,6.668078e+06,6266112.04,4.019663e+05,56664,10333
Kitchen,6.607540e+06,6174908.62,4.326313e+05,57140,10455
Decor,8.843480e+06,8285223.53,5.582568e+05,58142,10515
Indoor,1.141575e+07,10655257.93,7.604873e+05,59625,10674
Outdoor,1.783564e+07,16602807.44,1.232830e+06,57866,10552
Mobile Phones,3.331248e+07,31121754.47,2.190726e+06,52520,9526


### Check *discount*

In [17]:
sales_df["discount"].describe()

count    149101.000000
mean          0.249281
std           0.010440
min           0.220000
25%           0.240000
50%           0.250000
75%           0.260000
max           0.280000
Name: discount, dtype: float64

In [18]:
sales_df["discount"].value_counts().sort_index()

discount
0.22      778
0.23    10396
0.24    40041
0.25    55521
0.26    33283
0.27     8076
0.28     1006
Name: count, dtype: int64

In [43]:
discount_analysis = (
    sales_df
    .groupby("discount")
    .agg(
        transactions=("order_id", "count"),
        revenue=("revenue", "sum"),
        profit=("profit", "sum"),
        avg_profit=("profit", "mean")
    )
)

discount_analysis

,transactions,revenue,profit,avg_profit
discount,,,,
0.22,778,4.322916e+06,432456.88,555.857172
0.23,10396,3.433702e+07,19611074.36,1886.405768
0.24,40041,1.295161e+08,65879008.45,1645.288790
0.25,55521,1.337629e+08,47904542.27,862.818434
0.26,33283,7.048090e+07,3842499.98,115.449328
0.27,8076,2.100078e+07,937334.83,116.064243
0.28,1006,2.249947e+06,43868.79,43.607147


### Check *order status*

In [ ]:
sales_df.groupby("order_status")["order_id"].nunique()

order_status
Pending Payment      100
Received           25875
Shipped              756
Stocking              98
Name: order_id, dtype: int64

In [20]:
sales_df["order_priority"].value_counts()

order_priority
High        49566
Medium      25855
Critical    25119
Urgent      24325
Low         24236
Name: count, dtype: int64

### Check *payment method*

In [21]:
sales_df["payment_method"].value_counts()

payment_method
In-App Wallet    55592
Cash             31270
Credit Card      30979
Debit Card       30858
BNPL               402
Name: count, dtype: int64

## Data Preprocessing

In [46]:
sales_clean_df = sales_df.copy()

In [47]:
invalid_dates = sales_clean_df["year"] > 2026
print("Invalid dates:", invalid_dates.sum())

sales_clean_df = sales_clean_df[~invalid_dates]

Invalid dates: 3


*return_status* was removed because the column contained no valid observations.

In [48]:
sales_clean_df = sales_clean_df.drop(columns=["return_status"])

The rows with negative *cost_price_at_order_time* were removed because the negative cost is'nt meaningful in business.

In [49]:
sales_clean_df = sales_clean_df[sales_clean_df["cost_price_at_order_time"] >= 0]

In [72]:
print("Shape:", sales_clean_df.shape)

print("\nMissing values:")
print(sales_clean_df.isnull().sum().sum())

print("\nDuplicate rows:", sales_clean_df.duplicated().sum())

print("\nNegative cost:",
      (sales_clean_df["cost_price_at_order_time"] < 0).sum())

print("\nNegative profit:",
      (sales_clean_df["profit"] < 0).sum())

print("\nDate range:")
print(sales_clean_df["order_date"].min(),
      "→",
      sales_clean_df["order_date"].max())

Shape: (149096, 23)

Missing values:
0

Duplicate rows: 0

Negative cost: 0

Negative profit: 50391

Date range:
2020-01-01 00:00:00 → 2026-02-20 00:00:00


In [73]:
from pathlib import Path

output_path = Path("../data/sales_clean.csv")

output_path.parent.mkdir(parents=True, exist_ok=True)

sales_clean_df.to_csv(
    output_path,
    index=False
)

print(f"Saved to: {output_path}")

Saved to: ..\data\sales_clean.csv
